In [0]:
import logging
import yaml

dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("config_path", "")

environment = dbutils.widgets.get("environment")
CONFIG_PATH = dbutils.widgets.get("config_path")

logger = logging.getLogger("bronze_ingestion")
logger.setLevel(logging.INFO)

try:
    with open(CONFIG_PATH) as f:
        full_config = yaml.safe_load(f)

    config = full_config[environment]

    catalog = config["catalog"]
    bronze_schema = config["bronze_schema"]
    volumes_path = config["bronze_volumes_path"]

    sources = {
        config["consumption"]["source_file"]: f"{catalog}.{bronze_schema}.{config['consumption']['target_table']}",
        config["prices"]["source_file"]: f"{catalog}.{bronze_schema}.{config['prices']['target_table']}",
    }

    logger.info(f"[{environment}] Config loaded. Volumes path: {volumes_path}")
    logger.info(f"[{environment}] Sources: {sources}")

except Exception as e:
    logger.error(f"Failed to load config from {CONFIG_PATH} for environment '{environment}': {e}")
    raise

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date
from delta.tables import DeltaTable


def ingest_file(file_name, target_table):
    file_path = volumes_path + file_name

    df = (
        spark.read
        .option("multiline", "true")
        .json(file_path)
        .withColumn("source_filename", col("_metadata.file_path"))
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
        .dropDuplicates(["period", "series"])
    )

    if spark.catalog.tableExists(target_table):
        delta_table = DeltaTable.forName(spark, target_table)
        (
            delta_table.alias("target")
            .merge(
                df.alias("source"),
                "target.period = source.period AND target.series = source.series"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .withSchemaEvolution()
            .execute()
        )
        logger.info(f"[{target_table}] Merge completed.")
    else:
        (
            df.write
            .format("delta")
            .option("mergeSchema", "true")
            .saveAsTable(target_table)
        )
        logger.info(f"[{target_table}] Created table.")

In [0]:
for file_name, target_table in sources.items():
    ingest_file(file_name, target_table)